In [8]:
import pyspark.sql.functions as f
import pyspark.sql.types as t

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("meu_projeto")
    .master("local[*]") 
    .config("spark.sql.shuffle.partitions", "8")  
    .config("spark.driver.memory", "4g")          
    .config("spark.executor.memory", "4g")        
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  
    .getOrCreate()
)


25/04/21 11:43:32 WARN Utils: Your hostname, amaury-Nitro-AN515-52 resolves to a loopback address: 127.0.1.1; using 192.168.3.108 instead (on interface wlp0s20f3)
25/04/21 11:43:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/21 11:43:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_sisvan = spark.read.csv('/home/amaury/Documents/dissertacao/bases de dados/sisvan/sisvan_estado_nutricional_2023.csv', header=True, sep=';', encoding='iso-8859-1')

In [3]:
df_sisvan.show()

25/04/20 16:12:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+-----------------+-----------------+-----+-------------------+-------+------------+------------+--------------------+-------+-----------+--------------+------------------+------------------+---------------+--------------------+-----------------+--------------+-------+---------+------+----------------------+--------------------+--------------------+--------------------+----------------+--------------------+----------------+----------------------+---------------------+---------------------------+-----------------------+--------------------+
|   CO_ACOMPANHAMENTO|    CO_PESSOA_SISVAN|ST_PARTICIPA_ANDI|CO_MUNICIPIO_IBGE|SG_UF|       NO_MUNICIPIO|CO_CNES|NU_IDADE_ANO|NU_FASE_VIDA|        DS_FASE_VIDA|SG_SEXO|CO_RACA_COR|   DS_RACA_COR|CO_POVO_COMUNIDADE|DS_POVO_COMUNIDADE|CO_ESCOLARIDADE|     DS_ESCOLARIDADE|DT_ACOMPANHAMENTO|NU_COMPETENCIA|NU_PESO|NU_ALTURA|DS_IMC|DS_IMC_PRE_GESTACIONAL|        PESO X IDADE|       PESO X ALTURA| CRI. ALTURA X IDAD

In [4]:
df_sisvan.count()

52252848

### exploratória

- validação 

In [5]:
df_sisvan.groupBy('CO_ACOMPANHAMENTO').count().filter('count > 1').count()

0

In [ ]:
df_sisvan.groupBy('CO_PESSOA_SISVAN').count().filter('count > 1').count()

258568

In [ ]:
df_sisvan.select('CO_MUNICIPIO_IBGE').distinct().count() ## todo o brasil

5570

In [12]:
df_sisvan.select(f.max('DT_ACOMPANHAMENTO'), f.min('DT_ACOMPANHAMENTO')).show()

+----------------------+----------------------+
|max(DT_ACOMPANHAMENTO)|min(DT_ACOMPANHAMENTO)|
+----------------------+----------------------+
|            31/12/2023|            01/01/2023|
+----------------------+----------------------+



In [3]:
df_sisvan.select('CO_ESTADO_NUTRI_IDOSO').distinct().show()

+---------------------+
|CO_ESTADO_NUTRI_IDOSO|
+---------------------+
|            Sobrepeso|
| Adequado ou eutró...|
|           Baixo peso|
|                 NULL|
+---------------------+



In [4]:
df_sisvan.select('CO_ESTADO_NUTRI_ADULTO').distinct().show()

+----------------------+
|CO_ESTADO_NUTRI_ADULTO|
+----------------------+
|     Obesidade Grau II|
|      Obesidade Grau I|
|             Sobrepeso|
|  Adequado ou eutró...|
|    Obesidade Grau III|
|            Baixo peso|
|                  NULL|
+----------------------+



In [6]:
df_sisvan.select('CO_ESTADO_NUTRI_IMC_SEMGEST').distinct().show()

+---------------------------+
|CO_ESTADO_NUTRI_IMC_SEMGEST|
+---------------------------+
|                  Sobrepeso|
|                  Obesidade|
|       Adequado ou eutró...|
|                 Baixo peso|
|                       NULL|
+---------------------------+



In [13]:
(
    df_sisvan
    .withColumn(
        "aux",
        f.when(
            (f.col('CO_ESTADO_NUTRI_IDOSO').isNotNull()) &
            (
                (f.col('CO_ESTADO_NUTRI_ADULTO').isNotNull())
            ), 1
        ).otherwise(0)
    )
    .filter('aux == 1')
    .show()
)

+-----------------+----------------+-----------------+-----------------+-----+------------+-------+------------+------------+------------+-------+-----------+-----------+------------------+------------------+---------------+---------------+-----------------+--------------+-------+---------+------+----------------------+------------+-------------+-------------------+----------------+-------------------+----------------+----------------------+---------------------+---------------------------+-----------------------+--------------------+---+
|CO_ACOMPANHAMENTO|CO_PESSOA_SISVAN|ST_PARTICIPA_ANDI|CO_MUNICIPIO_IBGE|SG_UF|NO_MUNICIPIO|CO_CNES|NU_IDADE_ANO|NU_FASE_VIDA|DS_FASE_VIDA|SG_SEXO|CO_RACA_COR|DS_RACA_COR|CO_POVO_COMUNIDADE|DS_POVO_COMUNIDADE|CO_ESCOLARIDADE|DS_ESCOLARIDADE|DT_ACOMPANHAMENTO|NU_COMPETENCIA|NU_PESO|NU_ALTURA|DS_IMC|DS_IMC_PRE_GESTACIONAL|PESO X IDADE|PESO X ALTURA|CRI. ALTURA X IDADE|CRI. IMC X IDADE|ADO. ALTURA X IDADE|ADO. IMC X IDADE|CO_ESTADO_NUTRI_ADULTO|CO_ESTADO_

In [17]:
df_sisvan.columns

['CO_ACOMPANHAMENTO',
 'CO_PESSOA_SISVAN',
 'ST_PARTICIPA_ANDI',
 'CO_MUNICIPIO_IBGE',
 'SG_UF',
 'NO_MUNICIPIO',
 'CO_CNES',
 'NU_IDADE_ANO',
 'NU_FASE_VIDA',
 'DS_FASE_VIDA',
 'SG_SEXO',
 'CO_RACA_COR',
 'DS_RACA_COR',
 'CO_POVO_COMUNIDADE',
 'DS_POVO_COMUNIDADE',
 'CO_ESCOLARIDADE',
 'DS_ESCOLARIDADE',
 'DT_ACOMPANHAMENTO',
 'NU_COMPETENCIA',
 'NU_PESO',
 'NU_ALTURA',
 'DS_IMC',
 'DS_IMC_PRE_GESTACIONAL',
 'PESO X IDADE',
 'PESO X ALTURA',
 'CRI. ALTURA X IDADE',
 'CRI. IMC X IDADE',
 'ADO. ALTURA X IDADE',
 'ADO. IMC X IDADE',
 'CO_ESTADO_NUTRI_ADULTO',
 'CO_ESTADO_NUTRI_IDOSO',
 'CO_ESTADO_NUTRI_IMC_SEMGEST',
 'CO_SISTEMA_ORIGEM_ACOMP',
 'SISTEMA_ORIGEM_ACOMP']

In [22]:
(
    df_sisvan
    .withColumn(
        "aux",
        f.when(
            (f.col('CO_ESTADO_NUTRI_IDOSO').isNull()) &
            (
                (f.col('CO_ESTADO_NUTRI_ADULTO').isNull()) &
                (f.col("`ADO. IMC X IDADE`").isNull())
            ), 1
        ).otherwise(0)
    )
    .filter('aux == 1')
    .show()
)

+--------------------+--------------------+-----------------+-----------------+-----+-------------------+-------+------------+------------+--------------------+-------+-----------+--------------+------------------+------------------+---------------+---------------+-----------------+--------------+-------+---------+------+----------------------+--------------------+--------------------+--------------------+------------------+-------------------+----------------+----------------------+---------------------+---------------------------+-----------------------+--------------------+---+
|   CO_ACOMPANHAMENTO|    CO_PESSOA_SISVAN|ST_PARTICIPA_ANDI|CO_MUNICIPIO_IBGE|SG_UF|       NO_MUNICIPIO|CO_CNES|NU_IDADE_ANO|NU_FASE_VIDA|        DS_FASE_VIDA|SG_SEXO|CO_RACA_COR|   DS_RACA_COR|CO_POVO_COMUNIDADE|DS_POVO_COMUNIDADE|CO_ESCOLARIDADE|DS_ESCOLARIDADE|DT_ACOMPANHAMENTO|NU_COMPETENCIA|NU_PESO|NU_ALTURA|DS_IMC|DS_IMC_PRE_GESTACIONAL|        PESO X IDADE|       PESO X ALTURA| CRI. ALTURA X IDADE|  C